In [15]:
from pathlib import Path

base_dir = Path(
    "/Users/julian/PycharmProjects/PythonProject/MasterThesis/QAOA/Results/logs/ADAM"
)

patterns = [
    "*Exp4_v3_standardZeroAngles*",
    "*Exp8_standardZeroAngles_v3*",
]

paths = [
    csv
    for pattern in patterns
    for csv in base_dir.glob(f"{pattern}/sdp_cache_summary.csv")
]

print(paths)

[PosixPath('/Users/julian/PycharmProjects/PythonProject/MasterThesis/QAOA/Results/logs/ADAM/20260902_203134_pid1376936_Exp4_v3_standardZeroAngles/sdp_cache_summary.csv'), PosixPath('/Users/julian/PycharmProjects/PythonProject/MasterThesis/QAOA/Results/logs/ADAM/20260902_185944_pid906690_Exp4_v3_standardZeroAngles/sdp_cache_summary.csv'), PosixPath('/Users/julian/PycharmProjects/PythonProject/MasterThesis/QAOA/Results/logs/ADAM/20260902_185944_pid906687_Exp4_v3_standardZeroAngles/sdp_cache_summary.csv'), PosixPath('/Users/julian/PycharmProjects/PythonProject/MasterThesis/QAOA/Results/logs/ADAM/20260902_191029_pid1573207_Exp4_v3_standardZeroAngles/sdp_cache_summary.csv'), PosixPath('/Users/julian/PycharmProjects/PythonProject/MasterThesis/QAOA/Results/logs/ADAM/20260902_190226_pid1536162_Exp4_v3_standardZeroAngles/sdp_cache_summary.csv'), PosixPath('/Users/julian/PycharmProjects/PythonProject/MasterThesis/QAOA/Results/logs/ADAM/20260902_191006_pid1570592_Exp4_v3_standardZeroAngles/sdp_ca

In [16]:
import pandas as pd

results = []

tf176_l2_path = None

for path in paths:
    df = pd.read_csv(path)

    lasserre_level = df["lasserre_level"].iloc[0]
    graph_type = df["cache_dataset"].iloc[0]

    if (
        "trianglefree_connected_4to8vertices_176instances" in graph_type.lower()
        and lasserre_level == 2
    ):
        tf176_l2_path = path

    column_type = (
        "rounded_energy_over_exact_optimum"
        if lasserre_level == 1
        else "algorithm17_energy_over_exact_optimum"
    )

    results.append({
        "path": path,
        "graph_type": graph_type,
        "lasserre_level": lasserre_level,
        "average": df[column_type].mean(),
        "median": df[column_type].median(),
    })

print("tf176 L2 path:", tf176_l2_path)


# Same graph set together, Lasserre 1 before Lasserre 2
results.sort(key=lambda x: (x["graph_type"], x["lasserre_level"]))

for result in results:
    print(
        f"Graph type: {result['graph_type']}\n"
        f"Lasserre level: {result['lasserre_level']}\n"
        f"Average: {result['average']}\n"
        f"Median:  {result['median']}\n"
    )

tf176 L2 path: /Users/julian/PycharmProjects/PythonProject/MasterThesis/QAOA/Results/logs/ADAM/20260902_185944_pid906687_Exp4_v3_standardZeroAngles/sdp_cache_summary.csv
Graph type: 3regular_connected_74instances__filehash_688b5b4e
Lasserre level: 1
Average: 0.7938541106891163
Median:  0.7999999804313005

Graph type: 3regular_connected_74instances__filehash_688b5b4e
Lasserre level: 2
Average: 0.6745339220137286
Median:  0.6838232265187142

Graph type: complete_graphs_2to12vertices_11instances_AdjacencyList__filehash_1c7be33d
Lasserre level: 1
Average: 0.7271487464797354
Median:  0.7288284154660497

Graph type: complete_graphs_2to12vertices_11instances_AdjacencyList__filehash_1c7be33d
Lasserre level: 2
Average: 0.7255427788480071
Median:  0.7288286727279931

Graph type: cycle_graphs_3to12vertices_10instances_AdjacencyList__filehash_1a582721
Lasserre level: 1
Average: 0.6952286239228533
Median:  0.7068784320721053

Graph type: cycle_graphs_3to12vertices_10instances_AdjacencyList__filehas

In [20]:
from pathlib import Path
import pandas as pd

base_dir = Path(
    "/Users/julian/PycharmProjects/PythonProject/MasterThesis/QAOA/Results/logs/ADAM"
)

# All old Exp4_v2 tf176 runs
paths_exp4_v2 = list(
    base_dir.glob("*Exp4_v2_tf176*/sdp_cache_summary.csv")
)

optimal_path = tf176_l2_path
optimal_df = pd.read_csv(optimal_path)

results_exp4_v2 = []

for path in paths_exp4_v2:
    df = pd.read_csv(path)

    merged = df.merge(
        optimal_df[["graph_hash", "exact_optimal_energy"]],
        on="graph_hash",
        how="left",
        validate="many_to_one",
    )

    missing = merged["exact_optimal_energy"].isna()

    if missing.any():
        print(f"WARNING: {missing.sum()} unmatched graphs in:")
        print(path)
        print("Unmatched graph hashes:")
        print(merged.loc[missing, "graph_hash"].tolist())
        print()

    # These old Exp4_v2_tf176 files contain unnormalised Algorithm 17 energies
    merged["algorithm17_energy_over_exact_optimum"] = (
        (merged["algorithm17_actual_energy"] / 2)
        / merged["exact_optimal_energy"]
    )

    results_exp4_v2.append({
        "path": path,
        "graph_type": df["cache_dataset"].iloc[0],
        "lasserre_level": df["lasserre_level"].iloc[0],
        "average": merged["algorithm17_energy_over_exact_optimum"].mean(),
        "median": merged["algorithm17_energy_over_exact_optimum"].median(),
    })

results_exp4_v2.sort(
    key=lambda x: (
        x["graph_type"],
        x["lasserre_level"],
    )
)

for result in results_exp4_v2:
    print(
        f"Graph type: {result['graph_type']}\n"
        f"Lasserre level: {result['lasserre_level']}\n"
        f"Average: {result['average']}\n"
        f"Median:  {result['median']}\n"
    )

Graph type: triangleFree_connected_4to8vertices_176instances_AdjacencyList__filehash_279b7452
Lasserre level: 2
Average: 0.5854932804757986
Median:  0.5785112472292119

Graph type: triangleFree_connected_4to8vertices_176instances_AdjacencyList__filehash_279b7452
Lasserre level: 2
Average: 0.6154181160400117
Median:  0.6268405835820043

Graph type: triangleFree_connected_4to8vertices_176instances_AdjacencyList__filehash_279b7452
Lasserre level: 2
Average: 0.6059376342179098
Median:  0.612647518335447

Graph type: triangleFree_connected_4to8vertices_176instances_AdjacencyList__filehash_279b7452
Lasserre level: 2
Average: 0.6081968662475389
Median:  0.6073364689952221

Graph type: triangleFree_connected_4to8vertices_176instances_AdjacencyList__filehash_279b7452
Lasserre level: 2
Average: 0.5865591889000733
Median:  0.5821077192664856

Graph type: triangleFree_connected_4to8vertices_176instances_AdjacencyList__filehash_279b7452
Lasserre level: 2
Average: 0.6168213901141509
Median:  0.62494

In [19]:
import pandas as pd

# New tf176 L2 file that contains the exact optimal energies
optimal_df = pd.read_csv(tf176_l2_path)

# Keep exactly one exact value per graph
exact_values = (
    optimal_df[["graph_hash", "exact_optimal_energy"]]
    .drop_duplicates(subset="graph_hash")
)

print("Exact-value graphs:", len(exact_values))
print()

for path in paths_exp4_v2:
    df = pd.read_csv(path)

    merged = df.merge(
        exact_values,
        on="graph_hash",
        how="left",
        validate="many_to_one",
    )

    print(path.parent.name)
    print(f"Rows: {len(df)}")
    print(f"Matched: {merged['exact_optimal_energy'].notna().sum()}")
    print(f"Unmatched: {merged['exact_optimal_energy'].isna().sum()}")

    print(
        merged[
            [
                "graph_hash",
                "algorithm17_actual_energy",
                "exact_optimal_energy",
            ]
        ].head(10).to_string(index=False)
    )

    print()

Exact-value graphs: 176

20260901_143528_pid1286390_Exp4_v2_tf176_L2M2_scs_eps1e-10_gpseed42_alg17seed43
Rows: 174
Matched: 174
Unmatched: 0
      graph_hash  algorithm17_actual_energy  exact_optimal_energy
031f49f789e6245e                   2.845901              2.854638
faea56c01340eb7c                   7.013247              5.467166
f793499c6d3605b3                   6.357127              6.000000
9a09305d3fada0d0                   6.113112              5.427591
9d7a5b1d3a1c2c4c                   4.061153              3.451606
8b4362a019fdfffc                   4.118098              3.346997
5f9c64e19039c3b1                   6.965294              4.806220
6e3007103436a9dd                   5.704288              5.124933
74e713ee84278240                   4.754146              4.000000
96c1f2d4d001bdee                   3.857718              3.841530

20260901_110444_pid257353_Exp4_v2_tf176_L2M2_scs_eps1e-7_gpseed44_alg17seed45
Rows: 176
Matched: 176
Unmatched: 0
      graph_hash  

In [21]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    "/Users/julian/PycharmProjects/PythonProject/MasterThesis"
)

paths = [
    project_dir / "QAOA/Results/logs/ADAM/Exp4/Exp4_v2_standardZeroAngles/lr005_noheuristic/rand_78/qaoa_results_adam_20260829_133048_pid2057971_Exp4_v2_standardZeroAngles__rand_78_lr005_noheuristic_L2M2.csv",

    project_dir / "QAOA/Results/logs/ADAM/Exp4/Exp4_v2_standardZeroAngles/lr005_noheuristic/reg3_74/qaoa_results_adam_20260829_133048_pid2057981_Exp4_v2_standardZeroAngles__reg3_74_lr005_noheuristic_L2M2.csv",

    project_dir / "QAOA/Results/logs/ADAM/Exp4/Exp4_v2_standardZeroAngles/lr005_noheuristic/stress/qaoa_results_adam_20260829_133046_pid3689898_Exp4_v2_standardZeroAngles__stress_lr005_noheuristic_L2M2.csv",

    project_dir / "QAOA/Results/logs/ADAM/Exp4/Exp4_v2_standardZeroAngles/lr005_noheuristic/tf_176/qaoa_results_adam_20260829_133047_pid1402753_Exp4_v2_standardZeroAngles__tf_176_lr005_noheuristic_L1M1.csv",

    project_dir / "QAOA/Results/logs/ADAM/Exp8/Exp8_standardZeroAngles_v2/lr005_noheuristic/complete_2to12/qaoa_results_adam_20260827_203707_pid3511224_Exp8_standardZeroAngles_v2__complete_2to12_lr005_noheuristic_L2M2.csv",

    project_dir / "QAOA/Results/logs/ADAM/Exp8/Exp8_standardZeroAngles_v2/lr005_noheuristic/cycle_3to12/qaoa_results_adam_20260827_205813_pid3922153_Exp8_standardZeroAngles_v2__cycle_3to12_lr005_noheuristic_L2M2.csv",

    project_dir / "QAOA/Results/logs/ADAM/Exp8/Exp8_standardZeroAngles_v2/lr005_noheuristic/path_2to12/qaoa_results_adam_20260827_211929_pid136853_Exp8_standardZeroAngles_v2__path_2to12_lr005_noheuristic_L2M2.csv",
]

results = []

for path in paths:
    df = pd.read_csv(path)

    # Only p = 1, since algorithm17_actual_energy / optimal_result
    # is duplicated for the other QAOA depths
    df = df[df["p"] == 1].copy()

    # Approximation ratio -- these files are already correctly normalised
    df["algorithm17_energy_over_exact_optimum"] = (
        df["algorithm17_actual_energy"]
        / df["optimal_result"]
    )

    results.append({
        "graph_type": path.parent.name,
        "path": path,
        "n": len(df),
        "average": df["algorithm17_energy_over_exact_optimum"].mean(),
        "median": df["algorithm17_energy_over_exact_optimum"].median(),
    })

for result in results:
    print(
        f"Graph type: {result['graph_type']}\n"
        f"Rows (p=1): {result['n']}\n"
        f"Average: {result['average']}\n"
        f"Median:  {result['median']}\n"
    )

Graph type: rand_78
Rows (p=1): 34
Average: 0.6794408549407407
Median:  0.6799561486353982

Graph type: reg3_74
Rows (p=1): 74
Average: 0.6896226674573772
Median:  0.7002050793423189

Graph type: stress
Rows (p=1): 118
Average: 0.6797892284799129
Median:  0.6976027025433398

Graph type: tf_176
Rows (p=1): 176
Average: nan
Median:  nan

Graph type: complete_2to12
Rows (p=1): 9
Average: nan
Median:  nan

Graph type: cycle_3to12
Rows (p=1): 8
Average: nan
Median:  nan

Graph type: path_2to12
Rows (p=1): 9
Average: nan
Median:  nan

